
# CatBoost Classification on Pistachio Dataset  

Используем датасет **“16-Attribute Pistachio Dataset”** (Kaggle).  
В наборе 16 морфологических признаков фисташек и метка **`target`**  
(0 — Siirt, 1 — Kirmizi).

## Цель  
* Выбрать оптимальные параметры для **CatBoostClassifier**:  
  * доля валидационной выборки `test_size` ∈ {0.05, 0.06, 0.07};  
  * `random_state` ∈ {42, 1234, 2025}.  
* Для каждой пары параметров:  
  * обучить модель;  
  * измерить **hold‑out accuracy** и **k‑fold (StratifiedKFold, k = 5) accuracy**.  
* Найти лучшую комбинацию и предсказать метки для тестового множества.  

### Метрика  
$$
\text{Accuracy} \;=\; \frac{TP + TN}{TP+TN+FP+FN}.
$$



In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt

# Загрузка данных
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
display(train_df.head())


Train shape: (1288, 17), Test shape: (430, 16)


,area,perimeter,major_axis,minor_axis,eccentricity,eqdiasq,solidity,convex_area,extent,aspect_ratio,roundness,compactness,shapefactor_1,shapefactor_2,shapefactor_3,shapefactor_4,target
0,75516,1731.4840,411.7352,245.7620,0.8023,310.0806,0.9148,82546,0.7169,1.6753,0.3165,0.7531,0.0055,0.0033,0.5672,0.9502,1
1,98903,1374.4370,477.2451,269.7676,0.8249,354.8622,0.9585,103181,0.7679,1.7691,0.6579,0.7436,0.0048,0.0027,0.5529,0.9781,0
2,84746,1311.1570,482.7735,235.9040,0.8725,328.4843,0.9121,92914,0.7162,2.0465,0.6195,0.6804,0.0057,0.0028,0.4630,0.9474,1
3,98184,1463.1680,434.3769,292.6472,0.7390,353.5700,0.9543,102890,0.7316,1.4843,0.5763,0.8140,0.0044,0.0030,0.6625,0.9834,0
4,94170,1267.7271,440.1109,278.4162,0.7745,346.2672,0.9643,97656,0.6836,1.5808,0.7363,0.7868,0.0047,0.0030,0.6190,0.9785,0


In [6]:

test_sizes = [0.04, 0.05, 0.06, 0.07]
seeds = [42, 1234, 2025, 213, 2134, 413, 21324, 43, 4345]

X = train_df.drop(columns=['target'])
y = train_df['target']

results = []

for ts in test_sizes:
    for seed in seeds:
        # Hold‑out split
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=ts, random_state=seed, stratify=y
        )

        model = CatBoostClassifier(
            iterations=1000,
            learning_rate=0.03,
            depth=7,
            loss_function='Logloss',
            eval_metric='Accuracy',
            random_seed=seed,
            verbose=False
        )

        # Обучение
        model.fit(X_train, y_train)

        # Hold‑out accuracy
        y_pred = model.predict(X_val)
        hold_acc = accuracy_score(y_val, y_pred)

        # k‑fold CV (k=5)
        kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        cv_scores = cross_val_score(
            model, X, y, cv=kfold, scoring='accuracy', n_jobs=-1
        )
        cv_mean = cv_scores.mean()

        results.append({
            'test_size': ts,
            'seed': seed,
            'hold_out_acc': hold_acc,
            'cv_mean_acc': cv_mean
        })
        print(f"ts={ts:.2f}, seed={seed}: hold={hold_acc:.4f}, cv={cv_mean:.4f}")

results_df = pd.DataFrame(results).sort_values(
    by=['cv_mean_acc', 'hold_out_acc'], ascending=False
).reset_index(drop=True)

display(results_df)


ts=0.04, seed=42: hold=0.8077, cv=0.8625
ts=0.04, seed=1234: hold=0.8654, cv=0.8704
ts=0.04, seed=2025: hold=0.9038, cv=0.8735
ts=0.04, seed=213: hold=0.8269, cv=0.8688
ts=0.04, seed=2134: hold=0.8462, cv=0.8664
ts=0.04, seed=413: hold=0.8654, cv=0.8696
ts=0.04, seed=21324: hold=0.7885, cv=0.8634
ts=0.04, seed=43: hold=0.8077, cv=0.8649
ts=0.04, seed=4345: hold=0.8846, cv=0.8711
ts=0.05, seed=42: hold=0.8462, cv=0.8625
ts=0.05, seed=1234: hold=0.8308, cv=0.8704
ts=0.05, seed=2025: hold=0.8769, cv=0.8735
ts=0.05, seed=213: hold=0.8462, cv=0.8688
ts=0.05, seed=2134: hold=0.9231, cv=0.8664
ts=0.05, seed=413: hold=0.8615, cv=0.8696
ts=0.05, seed=21324: hold=0.8154, cv=0.8634
ts=0.05, seed=43: hold=0.8154, cv=0.8649
ts=0.05, seed=4345: hold=0.8923, cv=0.8711
ts=0.06, seed=42: hold=0.8846, cv=0.8625
ts=0.06, seed=1234: hold=0.8077, cv=0.8704
ts=0.06, seed=2025: hold=0.8974, cv=0.8735
ts=0.06, seed=213: hold=0.8718, cv=0.8688
ts=0.06, seed=2134: hold=0.9103, cv=0.8664
ts=0.06, seed=413: hold=

,test_size,seed,hold_out_acc,cv_mean_acc
0,0.04,2025,0.903846,0.873471
1,0.07,2025,0.901099,0.873471
2,0.06,2025,0.897436,0.873471
3,0.05,2025,0.876923,0.873471
4,0.06,4345,0.897436,0.871116
5,0.05,4345,0.892308,0.871116
6,0.04,4345,0.884615,0.871116
7,0.07,4345,0.879121,0.871116
8,0.04,1234,0.865385,0.870356
9,0.05,1234,0.830769,0.870356


In [7]:

# Лучшая комбинация
best = results_df.iloc[0]
BEST_TS = best['test_size']
BEST_SEED = int(best['seed'])
print(f"Best combination -> test_size={BEST_TS}, seed={BEST_SEED}")

# Финальное обучение на всех данных
final_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=7,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_seed=BEST_SEED,
    verbose=False
)
final_model.fit(X, y)

# Предсказания для теста
test_pred = final_model.predict(test_df)

answers = pd.DataFrame({'target': test_pred})
answers.to_csv('answers_catboost.csv', index=False, header=False)
answers.head()


Best combination -> test_size=0.04, seed=2025


,target
0,0
1,1
2,1
3,0
4,0



## Итоги  

* Наилучшая пара `(test_size, seed)` выбрана автоматически и отображена выше.  
* Модель обучена на **всех** тренировочных данных с этим `seed`.  
* Файл **`answers.csv`** с предсказаниями классов сохранён в текущей директории.  
* Для отправки на Kaggle загрузите `answers.csv`, убедившись, что столбец называется `target`.

